# RAG Demo: Retrieve, Augment, Generate

This notebook implements a tiny dependency-free RAG pipeline. It retrieves relevant documents with token overlap, builds a context window, and creates an extractive answer from that context.

In [7]:
import re
from collections.abc import Iterable


def tokenize(text: str) -> set[str]:
    return set(re.findall(r"[a-z0-9]+", text.lower()))


print("RAG demo initialized")

RAG demo initialized


In [8]:
documents = [
    {
        "id": "doc-1",
        "title": "Factorio blueprints",
        "text": "Factorio blueprints describe factory layouts and can be shared with other players.",
    },
    {
        "id": "doc-2",
        "title": "Retrieval augmented generation",
        "text": "RAG retrieves relevant context before generating an answer, which helps ground responses in source material.",
    },
    {
        "id": "doc-3",
        "title": "Python notebooks",
        "text": "Python notebooks are useful for interactive experiments because code and results stay together.",
    },
]

for document in documents:
    document["tokens"] = tokenize(document["text"])

documents

[{'id': 'doc-1',
  'title': 'Factorio blueprints',
  'text': 'Factorio blueprints describe factory layouts and can be shared with other players.',
  'tokens': {'and',
   'be',
   'blueprints',
   'can',
   'describe',
   'factorio',
   'factory',
   'layouts',
   'other',
   'players',
   'shared',
   'with'}},
 {'id': 'doc-2',
  'title': 'Retrieval augmented generation',
  'text': 'RAG retrieves relevant context before generating an answer, which helps ground responses in source material.',
  'tokens': {'an',
   'answer',
   'before',
   'context',
   'generating',
   'ground',
   'helps',
   'in',
   'material',
   'rag',
   'relevant',
   'responses',
   'retrieves',
   'source',
   'which'}},
 {'id': 'doc-3',
  'title': 'Python notebooks',
  'text': 'Python notebooks are useful for interactive experiments because code and results stay together.',
  'tokens': {'and',
   'are',
   'because',
   'code',
   'experiments',
   'for',
   'interactive',
   'notebooks',
   'python',
   'res

In [9]:
import hashlib
import math
from pathlib import Path

from pymilvus import DataType, MilvusClient


def embed(text: str, dimension: int = 8) -> list[float]:
    """Create a small deterministic embedding for this local demo."""
    vector = [0.0] * dimension
    for token in re.findall(r"[a-z0-9]+", text.lower()):
        digest = hashlib.sha256(token.encode("utf-8")).digest()
        index = int.from_bytes(digest[:4], "big") % dimension
        vector[index] += 1.0

    magnitude = math.sqrt(sum(value * value for value in vector))
    return [value / magnitude for value in vector] if magnitude else vector


database_path = (Path.cwd() / "miluvs_demo_vector.db").resolve()
client = MilvusClient(uri=str(database_path))
collection_name = "rag_documents"

if client.has_collection(collection_name):
    client.drop_collection(collection_name)

schema = client.create_schema(auto_id=False, enable_dynamic_field=False)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=8)
schema.add_field(field_name="title", datatype=DataType.VARCHAR, max_length=200)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=1000)

index_params = client.prepare_index_params()
index_params.add_index(field_name="vector", index_type="AUTOINDEX", metric_type="COSINE")
client.create_collection(collection_name, schema=schema, index_params=index_params)

rows = [
    {
        "id": index + 1,
        "vector": embed(document["text"]),
        "title": document["title"],
        "text": document["text"],
    }
    for index, document in enumerate(documents)
]
client.insert(collection_name=collection_name, data=rows)
print(f"Inserted {len(rows)} documents into {collection_name}")

Inserted 3 documents into rag_documents


In [10]:
query = "How does RAG improve answers?"
search_results = client.search(
    collection_name=collection_name,
    data=[embed(query)],
    limit=2,
    output_fields=["title", "text"],
)

print(f"Query: {query}")
print("\nRetrieved documents:")
for result in search_results[0]:
    entity = result["entity"]
    print(f"- score={result['distance']:.3f} | {entity['title']}")
    print(f"  {entity['text']}")

context = " ".join(result["entity"]["text"] for result in search_results[0])
print("\nContext for generation:")
print(context)

Query: How does RAG improve answers?

Retrieved documents:
- score=0.621 | Retrieval augmented generation
  RAG retrieves relevant context before generating an answer, which helps ground responses in source material.
- score=0.605 | Python notebooks
  Python notebooks are useful for interactive experiments because code and results stay together.

Context for generation:
RAG retrieves relevant context before generating an answer, which helps ground responses in source material. Python notebooks are useful for interactive experiments because code and results stay together.


In [11]:
def retrieve(query: str, documents: Iterable[dict], limit: int = 2) -> list[dict]:
    query_tokens = tokenize(query)
    ranked = []

    for document in documents:
        overlap = query_tokens & document["tokens"]
        if overlap:
            ranked.append((len(overlap), document))

    ranked.sort(key=lambda item: item[0], reverse=True)
    return [document for _, document in ranked[:limit]]


def answer(query: str, documents: Iterable[dict]) -> str:
    retrieved = retrieve(query, documents)
    if not retrieved:
        return "I could not find supporting context."

    context = " ".join(document["text"] for document in retrieved)
    query_tokens = tokenize(query)
    sentences = re.split(r"(?<=[.!?])\s+", context)
    best_sentence = max(sentences, key=lambda sentence: len(query_tokens & tokenize(sentence)))

    print("Retrieved context:")
    for document in retrieved:
        print(f"- [{document['id']}] {document['title']}: {document['text']}")
    return f"Answer: {best_sentence}"


query = "How does RAG improve answers?"
print(answer(query, documents))

Retrieved context:
- [doc-2] Retrieval augmented generation: RAG retrieves relevant context before generating an answer, which helps ground responses in source material.
Answer: RAG retrieves relevant context before generating an answer, which helps ground responses in source material.


In [12]:
import milvus_lite

print("milvus-lite is installed")
print("Version:", getattr(milvus_lite, "__version__", "available"))

milvus-lite is installed
Version: 2.5.1


In [ ]:
import re


# How tokenize works:
# 1. lower() makes matching case-insensitive
# 2. re.findall() keeps only word and number sequences
# 3. set() removes duplicate terms

def explain_tokenize(text: str) -> list[str]:
    normalized_text = text.lower()
    extracted_terms = re.findall(r"[a-z0-9]+", normalized_text)
    return extracted_terms

example_text = "RAG, RAG improves Answers in 2026!"
token_list = explain_tokenize(example_text)
print("Original:  ", example_text)
print("Token list:", token_list)
print("Token set: ", set(token_list))

Original:   RAG, RAG improves Answers in 2026!
Token list: ['rag', 'rag', 'improves', 'answers', 'in', '2026']
Token set:  {'improves', 'answers', 'rag', 'in', '2026'}


I0827 02:39:15.322243 4707939 chttp2_transport.cc:1369] unix:/var/folders/_2/9y93pxls66b3pscqspqtkt6w0000gn/T/tmpc6b_m4po_miluvs_demo_vector.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11, grpc_status:14}
E0827 02:39:15.322305 4707939 chttp2_transport.cc:1401] unix:/var/folders/_2/9y93pxls66b3pscqspqtkt6w0000gn/T/tmpc6b_m4po_miluvs_demo_vector.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms
